In [55]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import json
from datetime import datetime
import pandas as pd
import time

In [87]:
def grid_parser(driver, url):
    driver.get(url)
    time.sleep(1)

    rows = []
    iframes = driver.find_elements(By.CSS_SELECTOR, "iframe[src*='crosshare']")

    for idx in range(len(iframes)):
        try:
            iframes = driver.find_elements(By.CSS_SELECTOR, "iframe[src*='crosshare']")
            if idx >= len(iframes):
                break

            iframe = iframes[idx]
            src = iframe.get_attribute("src")
            print("src:", src)
            if not src or "crosshare" not in src:
                continue

            driver.switch_to.frame(iframe)
            print("a")
            time.sleep(0.5)

            script = driver.find_element(By.ID, "__NEXT_DATA__")
            json_text = script.get_attribute("innerHTML")

            data = json.loads(json_text)
            puzzle = data["props"]["pageProps"]["puzzle"]
            grid = puzzle["grid"]
            cols = puzzle["size"]["cols"]
            t = puzzle["publishTime"]
            dt = datetime.fromtimestamp(t / 1000).date()

            words = []

            # across words
            for i in range(0, len(grid), cols):
                row = grid[i:i + cols]
                row_str = "".join(row)
                for word in row_str.split("."):
                    if len(word) >= 2:
                        words.append(word)

            # down words
            for c in range(cols):
                col_chars = []
                for r in range(0, len(grid), cols):
                    col_chars.append(grid[r + c])

                col_str = "".join(col_chars)
                for word in col_str.split("."):
                    if len(word) >= 2:
                        words.append(word)

            for w in words:
                rows.append({"Date": dt, "Word": w})

        except Exception as e:
            print("iframe error:", url, idx, e)

        finally:
            try:
                driver.switch_to.default_content()
            except:
                pass

    return pd.DataFrame(rows)

In [57]:
def retrieveArticles(page, page_size):
    baseurl = f"https://www.cavalierdaily.com/section/quizzes?page={page}&per_page={page_size}"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(baseurl, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href = True):
        href = a["href"] 
        if "mini-crosswords" in href.lower():
            if href not in links:
                links.append(href)
    return links


In [83]:
df = (grid_parser(url = "https://www.cavalierdaily.com/article/2026/01/crossword-january-1?ct=content_open&cv=cbox_latest"))

src: https://crosshare.org/embed/iHLG0XdvB5J6mY7vcHmf/WiiyxL2cINd3FR9tuwaK94IvgsI2
a


In [84]:
df

,Date,Word
0,2025-12-26,CLUB
1,2025-12-26,NAB
2,2025-12-26,MAGMA
3,2025-12-26,HOBO
4,2025-12-26,POUR
...,...,...
75,2025-12-26,EGRESS
76,2025-12-26,ARMY
77,2025-12-26,ATM
78,2025-12-26,ASTRO


In [88]:
df = pd.DataFrame()
links = retrieveArticles(1, 200)
print(links)

options = Options()
options.page_load_strategy = "eager"
driver = webdriver.Chrome(options=options)

try:
    for i, l in enumerate(links):
        if i > 0 and i % 25 == 0:
            driver.quit()
            driver = webdriver.Chrome(options=options)
            print("restarted browser")

        print("link", l)
        try:
            articledf = grid_parser(driver, l)
            df = pd.concat([df, articledf], ignore_index=True)
        except Exception as e:
            print("failed on", l, e)
finally:
    driver.quit()

['https://www.cavalierdaily.com/article/2026/03/mini-crosswords-week-of-3-9?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/03/mini-crosswords-week-of-3-2?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-23?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-16?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-9?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-2?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/01/mini-crosswords-week-of-jan-26?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/01/mini-crosswords-week-of-jan-19?ct=content_open&cv=cbox_latest', 'https://www.cavalierdaily.com/article/2026/01/mini-crosswords-week-of-jan-12?ct=content_open&cv=cbox_latest', 'https:/

In [68]:
df["Word"].value_counts().head(20)

Word
SSS      4
YESES    3
LLAMA    3
ATARI    3
SANTA    3
NOCAP    3
NAIVE    3
SPASM    2
OLIOS    2
FILMS    2
FARES    2
FLOW     2
HAY      2
OFFOF    2
NIALL    2
ELRIO    2
ONE      2
ATHOS    2
CORN     2
MEOW     2
Name: count, dtype: int64

In [91]:
most_frequent_value = df['Word'].value_counts().idxmax()


In [90]:
df.shape

(1368, 2)

In [92]:
most_frequent_value
#deal with duplicates
#

'SSS'

In [101]:
df.loc[df["Word"]=="SANTA"]

,Date,Word
508,2025-12-22,SANTA
516,2025-12-22,SANTA
561,2025-12-07,SANTA


In [99]:
df["Word"].value_counts().head(20)

Word
OLIVE    3
ATARI    3
NAIVE    3
SANTA    3
SST      2
ONEAL    2
YSL      2
INANE    2
RICE     2
ALES     2
NURSE    2
MAN      2
ANGEL    2
DENIM    2
TESS     2
SCOTT    2
STEAL    2
PLAIN    2
CIDER    2
EVIAN    2
Name: count, dtype: int64

In [80]:
df.to_csv("words.csv")

In [98]:
words_to_remove = ["WXTJ", "MARIA", "ANARM", "STYES", "SSS", "MAS", "WANTS", "XRAYS", "TIRES", "JAMS",
                   "NELLY", "OLLIE", "CLANS", "AIMEE", "PEARS", "NOCAP", "ELLIE", "LLAMA", "LINER", "YESES",
                   "SPA", "PATCH", "OCHOA", "STORY", "ASSN", "SPOSA", "PACTS", "ATHOS", "CORN", "HAY",
                   "ONE", "FILMS", "FARES", "OLIOS", "FLOW", "OFFOF", "NIALL", "ELRIO", "MEOW", "SSS"]

for w in words_to_remove:
    idx = df.index[df["Word"] == w]
    if len(idx) > 0:
        df = df.drop(idx[0])